# Pacman - ThoresT

`Pacman.py` ist die Engine des Lehrers, unveraendert, plus unser `ThoresT`.
Die erste Zelle ist genau der Test aus `PacmanTest.ipynb`.

## 1. Der Original-Test des Lehrers

In [ ]:
from Pacman import Koordinaten, Direction, Position, Field

field = Field(15)
# print(field)
for i in range(100):
    print(" ")
    for pacman in field.pacmans:
        if pacman.alive:
            pacman.TurnOrMoveOrStill()
    print(field)

## 2. Dasselbe Spiel, aber nur das Ergebnis

Die Ausgabe oben ist 100 Spielfelder lang. Hier dieselbe Partie kompakt.

In [ ]:
import random
from Pacman import Field

random.seed(1)
field = Field(15)
me = field.pacmans[-1]

for turn in range(100):
    for pacman in field.pacmans:
        if pacman.alive:
            pacman.TurnOrMoveOrStill()

print(field)
print()
for pacman in sorted(field.pacmans, key=lambda p: -p.strength):
    mark = "  <-- wir" if pacman is me else ""
    status = "" if pacman.alive else "  (tot)"
    print(f"{pacman.name:<10s} {pacman.strength:6.0f}{status}{mark}")

rivals = [p.strength for p in field.pacmans if p is not me]
print()
print("ThoresT %.0f gegen besten Gegner %.0f -> %s"
      % (me.strength, max(rivals), "SIEG" if me.strength > max(rivals) else "verloren"))
print("Rechenzeit: %.2f ms/Zug, Fehler: %d"
      % (me.brain.total_ms / max(1, me.brain.turn), me.brain.faults))

## 3. Wie stark ist der Bot wirklich?

Viele Partien statt einer - eine einzelne Partie sagt bei diesem Spiel fast
nichts, die Kaempfe sind Wuerfelwuerfe.

In [ ]:
from superpac.pacman.arena import evaluate
from superpac.pacman.thorest import build_thorest
from superpac.pacman.opponents import build_opponents
import Pacman

ThoresT = build_thorest(Pacman.Pacman)
gegner = dict(build_opponents(Pacman.Pacman))

print("gegen die 5 Zufallsbots des Lehrers:")
print(" ", evaluate(ThoresT, games=30, label="ThoresT").row())
print(" ", evaluate(None,    games=30, label="leerer Stub").row())

### Der ehrlichere Test: gegen starke Gegner

Die Zufallsbots des Lehrers stehen ein Drittel der Zuege still. Sie zu
schlagen beweist wenig. Im Turnier stehen dort die Bots der anderen
Schueler - vermutlich ordentliche Ernte-Bots.

In [ ]:
H = gegner["harvester"]
mix = [gegner["harvester"], gegner["harvester"], gegner["hunter"],
       gegner["cautious"], gegner["sweeper"]]

print("gegen 5 Ernte-Bots (bei 6 gleich starken Spielern waeren 16.7% fair):")
print(" ", evaluate(ThoresT, games=30, label="ThoresT",   fillers=[H]*5).row())
print(" ", evaluate(H,       games=30, label="harvester", fillers=[H]*5).row())
print()
print("gemischtes Feld:")
print(" ", evaluate(ThoresT, games=30, label="ThoresT",   fillers=mix).row())
print(" ", evaluate(H,       games=30, label="harvester", fillers=mix).row())

## 4. Was der Bot denkt

Die wichtigste Regel des Spiels steckt in `Pacman._Move`: wen man angreift,
ist egal - **aus welcher Richtung** entscheidet alles.

In [ ]:
from superpac.pacman.rules import win_probability, NORTH, EAST, WEST

print("Bei gleicher Staerke gewinnt der Angreifer mit:")
print("  frontal (Ziel schaut mich an) %.1f%%" % (100*win_probability(10,10,EAST,WEST)))
print("  quer                          %.1f%%" % (100*win_probability(10,10,EAST,NORTH)))
print("  von hinten                    %.1f%%" % (100*win_probability(10,10,EAST,EAST)))
print()
print("Umgekehrt: wenn mich jemand angreift, bestimmt MEINE Blickrichtung")
print("seine Chance. Ihm entgegen zu schauen drueckt sie von 91% auf 50%.")

In [ ]:
import random
from superpac.pacman.agent import Brain
from superpac.pacman.thorest import build_thorest, execute
import Pacman

random.seed(4)
field = Field(15)
me = field.pacmans[-1]
brain = Brain(debug=True)

for turn in range(60):
    execute(me, brain.decide(me))
    for pacman in field.pacmans:
        if pacman is not me and pacman.alive:
            pacman.TurnOrMoveOrStill()

print(brain.log[-1])